# R26-DS-004 — baseline: TF-IDF + Logistic Regression (Colab)

**Layout**
- **Shared** (under `Research Project Files/`): preprocessed Parquet from `01_preprocess…` (`transaction_semantic_cache/`), and **`model_runs_compare.csv`** (one appended row per run for cross-model comparison).
- **This model only** (folder you created, e.g. `tf-idf_logistic-reg_model/`): `artifacts/` (`joblib`), `export/` (metrics JSON, run summary CSV, learned-weight summaries).

**Prereq:** Run your preprocess notebook once (e.g. `preprocess.ipynb` or repo `01_preprocess_paths_and_cache.ipynb`). **Cell 1 below** connects Google Drive on Colab if it is not mounted yet.

In [ ]:
# === CELL 1 — paths (Colab: connects Drive here if not already mounted) ===
from pathlib import Path


def _ensure_colab_drive_mounted() -> None:
    try:
        from google.colab import drive
    except ImportError:
        return  # local Jupyter — put data under a Path you set yourself
    if Path("/content/drive/MyDrive").is_dir():
        return
    print("Connecting to Google Drive… approve the browser prompt if it appears.")
    drive.mount("/content/drive")


def _drive_root() -> Path:
    for candidate in (Path("/content/drive/MyDrive"), Path("/content/drive/My drive")):
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        "Drive not found at /content/drive/MyDrive. On Colab: complete the mount prompt, then re-run this cell. "
        "Or use the folder icon > Mount Drive, then re-run."
    )


_ensure_colab_drive_mounted()
DRIVE_ROOT = _drive_root()

# --- Shared (all models) — same as preprocess notebook ---
RESEARCH_ROOT = DRIVE_ROOT / "Research Project Files"
CACHE_ROOT = RESEARCH_ROOT / "transaction_semantic_cache"
TRAIN_PQ = CACHE_ROOT / "train_preprocessed.parquet"
VAL_PQ = CACHE_ROOT / "val_preprocessed.parquet"
TEST_PQ = CACHE_ROOT / "test_preprocessed.parquet"
MANIFEST_JSON = CACHE_ROOT / "preprocess_manifest.json"

# --- This model only (edit folder name when you add another model notebook) ---
MODEL_FOLDER_NAME = "tf-idf_logistic-reg_model"
MODEL_RUN_ROOT = RESEARCH_ROOT / MODEL_FOLDER_NAME
ARTIFACT_DIR = MODEL_RUN_ROOT / "artifacts"
EXPORT_DIR = MODEL_RUN_ROOT / "export"

PIPELINE_JOBLIB = ARTIFACT_DIR / "tfidf_logreg_pipeline.joblib"
METRICS_JSON = EXPORT_DIR / "metrics.json"
RUN_SUMMARY_CSV = EXPORT_DIR / "run_summary.csv"
LOGREG_INTERCEPTS_CSV = EXPORT_DIR / "logistic_intercepts.csv"
COEF_TOP_FEATURES_CSV = EXPORT_DIR / "coefficients_top_features_by_class.csv"

# One file at Research root: append a row after each training run for later comparison
MODEL_RUNS_MASTER_CSV = RESEARCH_ROOT / "model_runs_compare.csv"

TOP_K_COEF_PER_CLASS = 150

print("DRIVE_ROOT (resolved)   =", DRIVE_ROOT)
print("RESEARCH_ROOT (shared)  =", RESEARCH_ROOT)
print("CACHE_ROOT              =", CACHE_ROOT)
print("MODEL_RUN_ROOT          =", MODEL_RUN_ROOT)
print("MODEL_RUNS_MASTER_CSV   =", MODEL_RUNS_MASTER_CSV)

In [ ]:
# === CELL 2 — verify cache files ===
missing = [p for p in (TRAIN_PQ, VAL_PQ, TEST_PQ, MANIFEST_JSON) if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "Missing preprocess outputs. Run preprocess.ipynb (or 01_preprocess…) first, or fix paths:\n"
        + "\n".join(str(p) for p in missing)
    )
print("OK — Parquet + manifest found.")

In [ ]:
# === CELL 3 — installs (once per runtime) ===
%pip install -q pandas pyarrow scikit-learn joblib numpy

In [ ]:
# === CELL 4 — imports ===
import json
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.pipeline import Pipeline

print("OK — imports")

In [ ]:
# === CELL 5 — load Parquet ===
train_df = pd.read_parquet(TRAIN_PQ)
val_df = pd.read_parquet(VAL_PQ)
test_df = pd.read_parquet(TEST_PQ)

for name, df in (("train", train_df), ("val", val_df), ("test", test_df)):
    if "text_primary" not in df.columns or "label" not in df.columns:
        raise ValueError(f"{name} missing text_primary or label")
    print(name, len(df))

print("OK — loaded")

In [ ]:
# === CELL 6 — define pipeline (TF-IDF + Logistic Regression) ===
pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True,
            ),
        ),
        (
            "clf",
            LogisticRegression(
                max_iter=200,
                class_weight="balanced",
                solver="saga",
                n_jobs=-1,
                random_state=42,
            ),
        ),
    ]
)

print("OK — pipeline defined")

In [ ]:
# === CELL 7 — fit on train, evaluate on val + test ===
X_train = train_df["text_primary"].astype(str)
y_train = train_df["label"].astype(str)

X_val = val_df["text_primary"].astype(str)
y_val = val_df["label"].astype(str)

X_test = test_df["text_primary"].astype(str)
y_test = test_df["label"].astype(str)

pipeline.fit(X_train, y_train)

val_pred = pipeline.predict(X_val)
test_pred = pipeline.predict(X_test)

val_f1 = f1_score(y_val, val_pred, average="macro", zero_division=0)
test_f1 = f1_score(y_test, test_pred, average="macro", zero_division=0)

print("macro-F1 val :", round(val_f1, 4))
print("macro-F1 test:", round(test_f1, 4))
print("\n--- classification report (test) ---")
print(classification_report(y_test, test_pred, zero_division=0))

print("OK — trained")

In [ ]:
# === CELL 8 — save joblib + metrics JSON under this model folder ===
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(pipeline, PIPELINE_JOBLIB)

metrics = {
    "model_id": "tfidf_logreg",
    "model_folder": MODEL_FOLDER_NAME,
    "macro_f1_val": float(val_f1),
    "macro_f1_test": float(test_f1),
    "n_train": int(len(train_df)),
    "n_val": int(len(val_df)),
    "n_test": int(len(test_df)),
    "pipeline_path": str(PIPELINE_JOBLIB),
}
METRICS_JSON.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print("OK — wrote")
print(PIPELINE_JOBLIB)
print(METRICS_JSON)

In [ ]:
# === CELL 9 — CSV exports: hyperparams + metrics, learned weights, master compare file ===

clf = pipeline.named_steps["clf"]
vec = pipeline.named_steps["tfidf"]
feat_names = vec.get_feature_names_out()
classes = clf.classes_
coef = clf.coef_
intercept = clf.intercept_


def _coef_class_labels():
    """Sklearn: multiclass coef rows align with classes_; binary has one row (positive class)."""
    if coef.shape[0] == len(classes):
        return [str(c) for c in classes]
    if coef.shape[0] == 1 and len(classes) == 2:
        return [str(classes[1])]
    return [f"coef_row_{i}" for i in range(coef.shape[0])]


coef_labels = _coef_class_labels()

# --- logistic_intercepts.csv (one row per intercept element) ---
int_rows = [
    {"class_label": coef_labels[i] if i < len(coef_labels) else f"row_{i}", "intercept": float(intercept[i])}
    for i in range(len(intercept))
]
pd.DataFrame(int_rows).to_csv(LOGREG_INTERCEPTS_CSV, index=False)

# --- coefficients_top_features_by_class.csv (top-K |weight| per coef row) ---
k = min(TOP_K_COEF_PER_CLASS, coef.shape[1])
coef_top = []
for ci, lab in enumerate(coef_labels):
    w = coef[ci]
    idx = np.argsort(np.abs(w))[-k:][::-1]
    for rank, j in enumerate(idx, start=1):
        coef_top.append(
            {
                "class_label": lab,
                "rank": rank,
                "feature": feat_names[j],
                "coefficient": float(w[j]),
            }
        )
pd.DataFrame(coef_top).to_csv(COEF_TOP_FEATURES_CSV, index=False)


def _scalar_for_csv(v):
    if v is None or isinstance(v, (bool, int, float)):
        return v
    if isinstance(v, str):
        return v[:800]
    if isinstance(v, (list, tuple, np.ndarray)):
        return str(v)[:800]
    return str(v)[:400]


flat = pipeline.get_params(deep=True)
hp_keys = [
    "tfidf__analyzer",
    "tfidf__ngram_range",
    "tfidf__min_df",
    "tfidf__max_df",
    "tfidf__sublinear_tf",
    "clf__solver",
    "clf__max_iter",
    "clf__class_weight",
    "clf__random_state",
    "clf__C",
]

weighted_f1_test = float(f1_score(y_test, test_pred, average="weighted", zero_division=0))

summary = {
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "model_id": "tfidf_logreg",
    "model_folder": MODEL_FOLDER_NAME,
    "n_vocab_features": int(len(feat_names)),
    "n_label_classes": int(len(classes)),
    "macro_f1_val": round(float(val_f1), 6),
    "macro_f1_test": round(float(test_f1), 6),
    "weighted_f1_test": round(weighted_f1_test, 6),
    "n_train": int(len(train_df)),
    "n_val": int(len(val_df)),
    "n_test": int(len(test_df)),
}
for hk in hp_keys:
    if hk in flat:
        summary[hk] = _scalar_for_csv(flat[hk])

summary_df = pd.DataFrame([summary])
summary_df.to_csv(RUN_SUMMARY_CSV, index=False)

# --- Append same row to Research Project Files/model_runs_compare.csv ---
if MODEL_RUNS_MASTER_CSV.is_file():
    old = pd.read_csv(MODEL_RUNS_MASTER_CSV)
    master_df = pd.concat([old, summary_df], ignore_index=True, sort=False)
else:
    master_df = summary_df
master_df.to_csv(MODEL_RUNS_MASTER_CSV, index=False)

print("OK — CSV exports (this model):")
print(RUN_SUMMARY_CSV)
print(LOGREG_INTERCEPTS_CSV)
print(COEF_TOP_FEATURES_CSV)
print("Appended / updated master:", MODEL_RUNS_MASTER_CSV, "(rows:", len(master_df), ")")